In [ ]:
# Import shared chat helper functions
from helper_functions import add_user_message, add_assistant_message, chat, DEFAULT_MODEL,client
# Initialize the conversation with an empty list of messages
messages = []

MODEL = "claude-haiku-4-5"

In [80]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": Format of the response "json" or "python" or "regex"
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    
    text = chat(messages, stop_sequences=["```"], model=MODEL, max_tokens=1000)
    return json.loads(text.strip())


In [81]:
dataset = generate_dataset()
print(json.dumps(dataset, indent=2))


with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[
  {
    "task": "Extract AWS S3 bucket names from CloudFormation template resource names using a pattern that matches 'AWS::S3::Bucket' resource definitions",
    "format": "regex",
    "solution_criteria": "Regex should accurately match S3 bucket resource type declarations and capture the logical resource ID. Should handle variations in whitespace and quotes."
  },
  {
    "task": "Parse an AWS IAM policy JSON string and extract all actions (Action field) that contain 's3:*' or 's3:GetObject' permissions",
    "format": "python",
    "solution_criteria": "Function should parse JSON policy, iterate through statements, and return a list of matching actions. Should handle both string and list formats for the Action field."
  },
  {
    "task": "Create a JSON configuration object for an AWS Lambda function that specifies runtime as 'python3.11', timeout of 60 seconds, memory of 512 MB, and environment variables for AWS_REGION and LOG_LEVEL",
    "format": "json",
    "solution_criteria"

In [ ]:
def run_prompt(test_case):
    """ Merges the prompt and the test case input, then returns the result"""
    
    prompt = f"""
    
    Please solve the following task:
    {test_case["task"]}
    
    * Respond only with python, json or a plain regex
    * Do not add any comment or commentary or explanation
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"], model=MODEL, max_tokens=1000, temperature=1.0)
    return output


In [82]:
# Grading by Model
def grade_by_model(test_case, output):
    """ Grades the output by asking the model to grade it. Returns the grade and feedback. """
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use for Evaluation:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"], model=MODEL, max_tokens=1000)
    return json.loads(eval_text)


In [70]:
import re
import ast
import json

def validate_json(text):
    """ Validates that the output is a well-formed JSON object. Returns 10 if valid, 0 otherwise. """
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
def validate_python(text):
    """ Validates that the output is a well-formed Python function. Returns 10  if valid, 0 otherwise. """
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    """ Validates that the output is a well-formed regular expression. Returns 10 if valid, 0 otherwise. """
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    """ Grades the syntax of the output based on the task type. Returns 10 if valid, 0 otherwise. """
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [76]:
def run_test_case(test_case):
    """ Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    # Calling model GRADER
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(output, test_case)
    # Combine the model score and the syntax score, 
    # Giving more weight to syntax score since it's an objective measure, while model score is subjective
    final_score = int(0.7 * model_score + 0.3 * syntax_score)
    return {
        "output": output,
        "test_case": test_case,
        "score": final_score,
        "reasoning": reasoning,
    }

In [83]:
from statistics import mean
def run_eval(dataset):
    """ Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result) 
        
    return results

In [86]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\nimport re\n\ndef extract_s3_bucket_names(template):\n    \"\"\"Extract AWS S3 bucket names from CloudFormation template.\"\"\"\n    if isinstance(template, str):\n        template = json.loads(template)\n    \n    s3_buckets = []\n    \n    if 'Resources' in template:\n        for resource_name, resource_config in template['Resources'].items():\n            if resource_config.get('Type') == 'AWS::S3::Bucket':\n                s3_buckets.append(resource_name)\n    \n    return s3_buckets\n",
    "test_case": {
      "task": "Extract AWS S3 bucket names from CloudFormation template resource names using a pattern that matches 'AWS::S3::Bucket' resource definitions",
      "format": "regex",
      "solution_criteria": "Regex should accurately match S3 bucket resource type declarations and capture the logical resource ID. Should handle variations in whitespace and quotes."
    },
    "score": 6,
    "reasoning": "While the solution effectively extracts S3

In [ ]:

average_score = mean(result["score"] for result in results)
print(f"Average Score: {average_score}")

Average Score: 7
